<a href="https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/datasources/wikidata/wikidata_fips_gnis_extract_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wikidata FIPS / GNIS Extraction → Google Drive

Streams the Wikidata JSON dump (~100 GB compressed), extracts all items with **P774 (FIPS)** or **P590 (GNIS)**, and saves a Parquet lookup table to your Google Drive.

**Why Colab?** Colab has much faster bandwidth to Wikimedia (~100–500 MB/s) than a home internet connection (~5–20 MB/s), so the same job that takes 10–20 hours locally finishes in **1–3 hours** here.

**Crash / disconnect safe:** checkpoints are written to Drive every 100k entities. If the session dies, re-open and run **Cell 5 (Resume)** — it skips already-scanned entities and picks up where it stopped.

## Steps
1. Mount Drive
2. Configure output folder
3. Run extraction  ← start here after the first setup
4. (If disconnected) Resume from checkpoint

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## 2 · Install dependencies & configure paths

In [ ]:
!pip install -q requests pyarrow pandas loguru tqdm

import bz2, json, os, time
from pathlib import Path
import pandas as pd
import requests
from loguru import logger
from tqdm.notebook import tqdm

# ── output directory on Drive ──────────────────────────────────────────────────
# Change this to any folder you like under /content/drive/MyDrive/
DRIVE_DIR = Path('/content/drive/MyDrive/CommunityOne/wikidata')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH    = DRIVE_DIR / 'fips_gnis_map.parquet'
CHECKPOINT_PATH = DRIVE_DIR / 'fips_gnis_map_checkpoint.parquet'
STATE_PATH      = DRIVE_DIR / 'fips_gnis_map_state.json'

DUMP_URL   = 'https://dumps.wikimedia.org/wikidatawiki/entities/latest-all.json.bz2'
# https://meta.wikimedia.org/wiki/User-Agent_policy — identify the script (avoid generic python-requests).
DUMP_HEADERS = {
    'User-Agent': 'OpenNavigator-fips-gnis-extract/1.0 (colab; +https://github.com/getcommunityone/open-navigator)',
}
PROP_FIPS  = 'P774'
PROP_GNIS  = 'P590'
CHUNK_SIZE = 1024 * 1024 * 4   # 4 MB decompress buffer
# Checkpoint every N entities — smaller than local script because Colab can
# disconnect without warning; Drive writes take ~2s so don't go below 50k.
FLUSH_EVERY = 100_000
LOG_EVERY   = 50_000

print(f'Output dir:  {DRIVE_DIR}')
print(f'Final file:  {PARQUET_PATH}')
print(f'Checkpoint:  {CHECKPOINT_PATH}')

## 3 · Helper functions

In [ ]:
# ── state ──────────────────────────────────────────────────────────────────────

def load_state():
    if STATE_PATH.exists():
        try:
            return json.loads(STATE_PATH.read_text())
        except Exception:
            pass
    return {'entities_scanned': 0, 'records_found': 0}

def save_state(entities_scanned, records_found):
    STATE_PATH.write_text(json.dumps({
        'entities_scanned': entities_scanned,
        'records_found':    records_found,
        'updated_at':       time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }, indent=2))

# ── extraction ─────────────────────────────────────────────────────────────────

def extract_claim_values(entity, prop):
    values = []
    for claim in entity.get('claims', {}).get(prop, []):
        try:
            val = claim['mainsnak']['datavalue']['value']
            if isinstance(val, str):
                values.append(val.strip())
        except (KeyError, TypeError):
            continue
    return values

def entity_to_records(entity):
    if entity.get('type') != 'item':
        return []
    qid       = entity.get('id', '')
    fips_vals = extract_claim_values(entity, PROP_FIPS)
    gnis_vals = extract_claim_values(entity, PROP_GNIS)
    if not fips_vals and not gnis_vals:
        return []
    label    = entity.get('labels', {}).get('en', {}).get('value', '')
    modified = entity.get('lastrevid')
    rows = []
    for fips in fips_vals:
        rows.append({'qid': qid, 'label': label, 'fips': fips,
                     'gnis': gnis_vals[0] if gnis_vals else None,
                     'modified': modified, 'source': 'fips'})
    if not fips_vals:
        for gnis in gnis_vals:
            rows.append({'qid': qid, 'label': label, 'fips': None,
                         'gnis': gnis, 'modified': modified, 'source': 'gnis'})
    return rows

# ── streaming ──────────────────────────────────────────────────────────────────

def stream_wikidata_dump(url, skip=0):
    """
    Stream the bz2 dump. Yields parsed entity dicts.
    skip: fast-forward past this many already-scanned entities (for resume).
    """
    scanned = 0
    with requests.get(url, stream=True, timeout=300, headers=DUMP_HEADERS) as resp:
        resp.raise_for_status()
        decompressor = bz2.BZ2Decompressor()
        buffer = b''
        for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
            buffer += decompressor.decompress(chunk)
            lines = buffer.split(b'\n')
            buffer = lines[-1]
            for line in lines[:-1]:
                line = line.strip()
                if not line or line in (b'[', b']'):
                    continue
                line = line.rstrip(b',')
                try:
                    entity = json.loads(line)
                except json.JSONDecodeError:
                    continue
                scanned += 1
                if scanned <= skip:
                    continue
                yield entity

# ── checkpoint ─────────────────────────────────────────────────────────────────

def flush_checkpoint(records, entities_scanned):
    pd.DataFrame(records).to_parquet(CHECKPOINT_PATH, index=False)
    save_state(entities_scanned, len(records))
    print(f'  checkpoint: {entities_scanned:,} entities scanned, {len(records):,} matches')

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        df = pd.read_parquet(CHECKPOINT_PATH)
        print(f'Loaded {len(df):,} records from checkpoint.')
        return df.to_dict('records')
    return []

print('Functions loaded.')

## 4 · Run extraction (fresh start)

Expected time: **1–3 hours** on Colab (CPU-bound, not network-bound).

If the session disconnects, skip to **Cell 5 (Resume)** instead.

In [ ]:
if STATE_PATH.exists():
    existing = load_state()
    if existing.get('entities_scanned', 0):
        raise RuntimeError(
            f"Found a checkpoint at {existing['entities_scanned']:,} entities. "
            "Run Cell 5 (Resume) instead, or delete the checkpoint files on Drive to start fresh."
        )

records       = []
entity_count  = 0
start         = time.time()

pbar = tqdm(desc='Entities scanned', unit=' entities', mininterval=5)

for entity in stream_wikidata_dump(DUMP_URL):
    entity_count += 1
    records.extend(entity_to_records(entity))

    pbar.update(1)
    if entity_count % LOG_EVERY == 0:
        elapsed = time.time() - start
        rate    = entity_count / elapsed
        pbar.set_postfix(matches=len(records), rate=f'{rate/1000:.0f}k/s',
                         elapsed=f'{elapsed/60:.0f}m')

    if entity_count % FLUSH_EVERY == 0:
        flush_checkpoint(records, entity_count)

pbar.close()

elapsed = time.time() - start
print(f'\nScan complete: {entity_count:,} entities, {len(records):,} matches in {elapsed/3600:.1f}h')

df_final = pd.DataFrame(records)
df_final.to_parquet(PARQUET_PATH, index=False)
print(f'Saved {len(df_final):,} rows → {PARQUET_PATH}')

# Clean up checkpoint files
for p in (CHECKPOINT_PATH, STATE_PATH):
    if p.exists(): p.unlink()
print('Done. Checkpoint files cleaned up.')

## 5 · Resume after disconnect

If Cell 4 was interrupted, run this cell instead. It reads the last checkpoint from Drive, fast-forwards past already-scanned entities, then continues from where it stopped.

In [ ]:
state = load_state()
skip_entities = state.get('entities_scanned', 0)

if not skip_entities:
    print('No checkpoint found — run Cell 4 (fresh start) instead.')
else:
    print(f'Resuming from entity {skip_entities:,} '
          f'({state.get("records_found", 0):,} matches already saved).')
    print('Fast-forwarding through already-scanned entities (no extraction — just counting)...')

    records      = load_checkpoint()
    entity_count = skip_entities
    start        = time.time()

    pbar = tqdm(desc='Entities (post-resume)', unit=' entities', mininterval=5)

    for entity in stream_wikidata_dump(DUMP_URL, skip=skip_entities):
        entity_count += 1
        records.extend(entity_to_records(entity))

        pbar.update(1)
        if entity_count % LOG_EVERY == 0:
            elapsed = time.time() - start
            rate    = (entity_count - skip_entities) / elapsed
            pbar.set_postfix(total_matches=len(records), rate=f'{rate/1000:.0f}k/s')

        if entity_count % FLUSH_EVERY == 0:
            flush_checkpoint(records, entity_count)

    pbar.close()

    elapsed = time.time() - start
    print(f'\nResume complete: {entity_count:,} total entities, '
          f'{len(records):,} matches, +{elapsed/60:.0f}m this session')

    df_final = pd.DataFrame(records)
    df_final.to_parquet(PARQUET_PATH, index=False)
    print(f'Saved {len(df_final):,} rows → {PARQUET_PATH}')

    for p in (CHECKPOINT_PATH, STATE_PATH):
        if p.exists(): p.unlink()
    print('Done. Checkpoint files cleaned up.')

## 6 · Inspect results

In [ ]:
df = pd.read_parquet(PARQUET_PATH)
print(f'Rows:    {len(df):,}')
print(f'FIPS:    {df["fips"].notna().sum():,}')
print(f'GNIS:    {df["gnis"].notna().sum():,}')
print(f'File:    {PARQUET_PATH.stat().st_size / 1e6:.1f} MB')
df.head(10)

## 7 · Copy Parquet back to local machine (optional)

The Parquet file lives in your Drive at `MyDrive/CommunityOne/wikidata/fips_gnis_map.parquet`.
You can download it from the Drive web UI, or run the cell below to copy it into this repo if you cloned it to Colab.

In [ ]:
# Only needed if you cloned the repo into Colab and want the file inside it.
# Otherwise just download from Drive UI.
import shutil

REPO_CACHE = Path('/content/open-navigator/data/cache/wikidata')
if REPO_CACHE.exists():
    dest = REPO_CACHE / 'fips_gnis_map.parquet'
    shutil.copy2(PARQUET_PATH, dest)
    print(f'Copied → {dest}')
else:
    print('Repo not cloned here — download the file from Drive UI instead.')
    print(f'Drive path: {PARQUET_PATH}')